In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]
plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["figure.constrained_layout.use"] = True

import numpy as np
from treebeard.dataset import get_jets, get_events
import os
import pandas as pd
from scipy.stats import entropy, pearsonr, spearmanr
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.metrics import roc_curve, auc
import torch
from tqdm import tqdm
import json
from treebeard.SDT import SDT

In [ ]:
input_dim  = 57
output_dim = 2
depth      = 4
lamda      = 1e-5
device     = torch.device('cpu')

In [ ]:
home  = '/eos/home-i00/d/dhnaik' 
data_dir = 'C2V_event_training_data'
model_dir = os.path.join(home, 'SDT/path_output/EVENT_C2V')
test_dir = os.path.join(home, 'SDT/test_outputs/EVENT_C2V')

input_directory = os.path.join(home, data_dir)
with open(input_directory + '/meta_dict_corrected.json') as f:
    meta_dict = json.load(f)

feature_dict = meta_dict['input_vars']
class_labels = meta_dict['class_labels']

X_features = np.load(input_directory+'/train/X_features.npy')
y_labels   = np.load(input_directory+'/train/y_label.npy')

In [ ]:
jet_feature_list = ['L1T_JetPuppiAK4_PT','L1T_JetPuppiAK4_Eta','L1T_JetPuppiAK4_Phi']
muon_feature_list = ['L1T_MuonTight_PT','L1T_MuonTight_Eta','L1T_MuonTight_Phi']
electron_feature_list = ['L1T_Electron_PT','L1T_Electron_Eta','L1T_Electron_Phi']
met_feature_list = ['L1T_PUPPIMET_MET','L1T_PUPPIMET_Eta','L1T_PUPPIMET_Phi']

max_number_of_jets = 10
max_number_of_muons = 4
max_number_of_electrons = 4

top_x_jets = [feature + str(i) for i in range(max_number_of_jets) for feature in jet_feature_list ]
top_x_muons = [feature + str(i) for i in range(max_number_of_muons) for feature in muon_feature_list]
top_x_electrons = [feature + str(i) for i in range(max_number_of_electrons) for feature in electron_feature_list]
all_columns = top_x_jets + top_x_muons + top_x_electrons + met_feature_list

In [ ]:
tree_d6 = SDT(input_dim, output_dim, depth=6, lamda=lamda, use_cuda=False).to(device, non_blocking=True)
tree_d6.load_state_dict(torch.load('/eos/user/d/dhnaik/SDT/path_output/EVENT_C2V/sdt_flat.pth', map_location='cpu', weights_only=True))
tree_d6.eval()

tree_d4 = SDT(input_dim, output_dim, depth=4, lamda=lamda, use_cuda=False).to(device, non_blocking=True)
tree_d4.load_state_dict(torch.load('/eos/user/d/dhnaik/SDT/path_output/EVENT_C2V/sdt_depth4.pth', map_location='cpu', weights_only=True))
tree_d4.eval()

tree_d3 = SDT(input_dim, output_dim, depth=3, lamda=lamda, use_cuda=False).to(device, non_blocking=True)
tree_d3.load_state_dict(torch.load('/eos/user/d/dhnaik/SDT/path_output/EVENT_C2V/sdt_depth3.pth', map_location='cpu', weights_only=True))
tree_d3.eval()

tree_d2 = SDT(input_dim, output_dim, depth=2, lamda=lamda, use_cuda=False).to(device, non_blocking=True)
tree_d2.load_state_dict(torch.load('/eos/user/d/dhnaik/SDT/path_output/EVENT_C2V/sdt_depth2.pth', map_location='cpu', weights_only=True))
tree_d2.eval()

W_full = tree_d4.inner_nodes[0].weight.detach().cpu().numpy()
bias_d4 = W_full[:, 0]
W_d4 = W_full[:, 1:]
beta_d4 = torch.clamp(tree_d4.beta, max=5.0).detach().cpu().numpy()
print('depth = 4')
print("Num internal nodes:", tree_d4.internal_node_num_)
print("W shape:", W_d4.shape, "bias shape:", bias_d4.shape, "beta shape:", beta_d4.shape)

In [ ]:
X_train = np.load('/eos/home-i00/d/dhnaik/C2V_event_training_data/event_cache/X_train_j10_m4_e4.npy')
X_train_tensor = torch.FloatTensor(X_train)

X_test = np.load('/eos/home-i00/d/dhnaik/C2V_event_training_data/event_cache/X_test_j10_m4_e4.npy')
X_test_tensor = torch.FloatTensor(X_test)

In [ ]:
tree = tree_d6

X_train_aug = tree._data_augment(X_train_tensor.to(tree.device))
X_test_aug = tree._data_augment(X_test_tensor.to(tree.device))

beta_clamped = torch.clamp(tree.beta, max=5.0)

logits_orig = beta_clamped[None, :] * (X_test_aug @ tree.inner_nodes[0].weight.T)

W_eff = beta_clamped[:, None] * tree.inner_nodes[0].weight  # this one IS correct — W is (internal_node_num_, input_dim+1), beta needs to broadcast per-row, so [:, None] is right here
logits_folded = X_test_aug @ W_eff.T

assert torch.allclose(logits_orig, logits_folded, atol=1e-4)
print("Fold verified — outputs match")

W_eff_np = W_eff.detach().numpy().flatten()
print('max = ',W_eff_np.max())
print('min =',W_eff_np.min())

int_bits = np.ceil(np.log2(max(abs(W_eff_np)))) + 1
print(f'\nint bits = {int_bits}')

res = 0.01
frac_bits = np.ceil(np.log2(1 / res))
print(f'frac bits = {frac_bits}')

In [ ]:
logits_np = logits_folded.detach().numpy().flatten()   # or logits_orig, same values
print('logits max =', logits_np.max())
print('logits min =', logits_np.min())

acc_max_abs = max(abs(logits_np.max()), abs(logits_np.min()))
acc_int_bits = np.ceil(np.log2(acc_max_abs)) + 1
print(f'acc int_bits = {acc_int_bits}')
plt.figure()
plt.grid(alpha=0.2)
plt.hist(logits_np, histtype='step', bins=50, color='#AF719D')
plt.hist(logits_np, histtype='stepfilled', alpha=0.3, bins=50, color='#AF719D')
plt.title('Node Logit Distribution')
plt.yscale('log')
plt.xlabel('Node Logits')
plt.ylabel('frequency')
plt.show()

leaf_logits = tree.leaf_logits.detach().numpy()
plt.figure()
plt.grid(alpha=0.2)
plt.title('Leaf Logit Distribution')
plt.hist(leaf_logits.flatten(), histtype='step', bins=12, color='#AF719D')
plt.hist(leaf_logits.flatten(), histtype='stepfilled', alpha=0.3, bins=12, color='#AF719D')
plt.yscale('log')
plt.xlabel('Leaf Logits')
plt.ylabel('Frequency')
plt.show()

In [ ]:
from fxpmath import Fxp

def quantize_ap_fixed(x, W, I):
    """
    Simulates Xilinx C++ ap_fixed<W, I> with default AP_TRN (truncation toward minus infinity) 
    and AP_WRAP (overflow wrap around).
    """
    n_frac = W - I
    scale = 2 ** n_frac
    
    # Range for signed 2's complement W-bit number
    min_val = -(2 ** (W - 1))
    max_val = 2 ** (W - 1) - 1
    
    # Default C++ ap_fixed assignment behavior is floor truncation (AP_TRN)
    x_scaled = np.floor(x * scale)
    
    # Handle two's complement overflow wrap-around (AP_WRAP)
    # If you want saturation (AP_SAT), use np.clip(x_scaled, min_val, max_val)
    range_span = 2 ** W
    x_wrapped = (x_scaled - min_val) % range_span + min_val
    
    return x_wrapped / scale

def quantize_ap_ufixed(x, W, I):
    """
    Simulates Xilinx C++ ap_ufixed<W, I> with default AP_TRN (truncation toward minus infinity)
    and AP_WRAP (overflow wrap around).
    """
    n_frac = W - I
    scale = 2 ** n_frac

    # Range for unsigned W-bit number
    min_val = 0
    max_val = 2 ** W - 1

    # Default truncation toward minus infinity (AP_TRN) -- same as signed case
    x_scaled = np.floor(x * scale)

    # Unsigned wrap-around: simple modulo into [0, 2^W - 1], no offset needed
    range_span = 2 ** W
    x_wrapped = x_scaled % range_span

    if np.any(x < 0):
        print(f"Warning: {np.sum(x < 0)} negative values passed to quantize_ap_ufixed — will wrap incorrectly")

    return x_wrapped / scale

W, I = 12, 4
w = W_eff_np
w_fp = quantize_ap_fixed(W_eff_np, W, I)
w_fxp = Fxp(W_eff_np, signed=True, n_word=W, n_frac=W-I, rounding='floor', overflow='wrap')

i=0
print('raw weight =       ',     w[i])
print('fxp from scratch = ',  w_fp[i])
print('fxpmath =          ', w_fxp[i])

In [ ]:
X_test.shape

In [ ]:
import timeit

# pass arguments via lambda to avoid scope issues
elapsed_time = timeit.timeit(
    lambda: Fxp(
        X_test,
        signed=True,
        n_word=W,
        n_frac=W - I,
        rounding="floor",
        overflow="wrap",
    ),
    number=10,  # Runs function 100 times
)

print(f"Average time per run: {(elapsed_time / 10) * 1000:.3f} ms")

In [ ]:
pt = X_test[:,0::3][:,:18]
met = X_test[:,0::3][:,-1]
eta = X_test[:,1::3]
phi = X_test[:,2::3]
print(f'minimum pt : {pt.min()}')
print(f'maximum pt : {pt.max()}')
print(f'\nminimum met : {met.min()}')
print(f'maximum met : {met.max()}')
print(f'\nminimum eta : {eta.min()}')
print(f'maximum eta : {eta.max()}')
print(f'\nminimum phi : {phi.min()}')
print(f'maximum phi : {phi.max()}')

In [ ]:
int_bits = np.ceil(np.log2(abs(met.max()))) + 1
print(f'int bits = {int_bits}')

In [ ]:
pt_fxp  = Fxp(pt,  signed=False, n_word=14, n_frac=2, rounding='floor', overflow='wrap')
met_fxp = Fxp(met, signed=False, n_word=14, n_frac=2, rounding='floor', overflow='wrap')
eta_fxp = Fxp(eta, signed=True,  n_word=12, n_frac=7, rounding='floor', overflow='wrap')
phi_fxp = Fxp(phi, signed=True,  n_word=12, n_frac=9, rounding='floor', overflow='wrap')

print(f'       pt_fxp     = {pt_fxp[:5,0]}')
print(f'scaled pt_fxp.val = {pt_fxp[:5,0].val * 2**-2}')
print(f'       pt         = {pt[:5,0]}')

ok need to add overflow on multiplication handling, implement into `dataset.py`, then figure out sigmoid LUT.  
`_mu` and `leaf_probs` should be straightforward - bounded between [0,1]  
forward pass w/ quantised weights seems fine, just need to fix `_mu` and `leaf_probs` - should also have unquantised vals returned in same function  
so much work ough

### `analysis`

In [ ]:
def plot_sweep_old(df: pd.DataFrame, axis: str = "w_n_word", requant_every_layer: bool = True):
    """
    Accuracy vs a chosen bit-width axis, one line per depth, taking the best
    acc_quant at each x-value (max over all other swept params), filtered to
    one requant_every_layer setting.

    axis: one of
        "w_n_word", "w_n_frac", "w_n_int"    (weight bit-width variants)
        "mu_n_word", "mu_n_frac", "mu_n_int" (mu bit-width variants)
    """
    valid_axes = {"w_n_word", "w_n_frac", "w_n_int", "mu_n_word", "mu_n_frac", "mu_n_int"}
    assert axis in valid_axes, f"axis must be one of {valid_axes}, got {axis!r}"

    sub = df[df["requant_every_layer"] == requant_every_layer].copy()

    # derive n_int columns (n_word - n_frac) for weight and mu
    sub["w_n_int"] = sub["w_n_word"] - sub["w_n_frac"]
    sub["mu_n_int"] = sub["mu_n_word"] - sub["mu_n_frac"]

    axis_labels = {
        "w_n_word":  "Weight bit-width (n_word)",
        "w_n_frac":  "Weight fractional bits (n_frac)",
        "w_n_int":   "Weight integer bits (n_int)",
        "mu_n_word": "Mu bit-width (n_word)",
        "mu_n_frac": "Mu fractional bits (n_frac)",
        "mu_n_int":  "Mu integer bits (n_int)",
    }

    plt.figure(figsize=(10, 6))
    for depth_name, group in sub.groupby("depth"):
        best = group.groupby(axis)["acc_quant"].max().reset_index()
        best = best.sort_values(axis)
        plt.plot(best[axis], best["acc_quant"], marker="o", label=depth_name)

    baselines = sub.groupby("depth")["acc_float"].first()
    for depth_name, acc in baselines.items():
        plt.axhline(acc, linestyle="--", alpha=0.3)

    plt.xlabel(axis_labels[axis])
    plt.ylabel("Test accuracy")
    plt.title(f"Quantization sweep vs {axis_labels[axis]}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(f"quant_sweep_plot_{axis}.png", dpi=150)
    plt.show()

In [ ]:
def plot_sweep(df: pd.DataFrame, label: str, axis: str = "w_n_word", requant_every_layer: bool = True):
    """
    Accuracy vs a chosen bit-width axis, one line per depth, taking the best
    acc_quant at each x-value (max over all other swept params), filtered to
    one requant_every_layer setting.
    axis: one of
        "w_n_word", "w_n_frac", "w_n_int"    (weight bit-width variants)
        "mu_n_word", "mu_n_frac", "mu_n_int" (mu bit-width variants)

    X-axis ticks are labeled in ap_fixed format <n_word, n_int>.
    """
    valid_axes = {"w_n_word", "w_n_frac", "w_n_int",
                  "mu_n_word", "mu_n_frac", "mu_n_int", 
                  "acc_n_word", "acc_n_frac", "acc_n_int",
                  "leaf_n_word", "leaf_n_frac", "leaf_n_int"}
    
    assert axis in valid_axes, f"axis must be one of {valid_axes}, got {axis!r}"
    
    if axis.startswith("w_"):
        prefix = "w_"
    elif axis.startswith("mu_"):
        prefix = "mu_"
    elif axis.startswith("acc_"):
        prefix = "acc_"
    else:
        prefix = "leaf_"
        
    word_col = f"{prefix}n_word"
    frac_col = f"{prefix}n_frac"
    int_col = f"{prefix}n_int"

    sub = df[df["requant_every_layer"] == requant_every_layer].copy()
    sub["w_n_int"] = sub["w_n_word"] - sub["w_n_frac"]
    sub["mu_n_int"] = sub["mu_n_word"] - sub["mu_n_frac"]
    sub["acc_n_int"] = sub["acc_n_word"] - sub["acc_n_frac"]
    sub["leaf_n_int"] = sub["leaf_n_word"] - sub["leaf_n_frac"]

    axis_labels = {
        "w_n_word":  "Weight bit-width",
        "w_n_frac":  "Weight fractional bits (n_frac)",
        "w_n_int":   "Weight integer bits (n_int)",
        "mu_n_word": "Mu bit-width",
        "mu_n_frac": "Mu fractional bits (n_frac)",
        "mu_n_int":  "Mu integer bits (n_int)",
        "acc_n_word":  "Node Logit bit-width",
        "acc_n_frac":  "Node Logit integer bits (n_int)",
        "acc_n_int":   "Node Logit integer bits (n_int)",
        "leaf_n_word":  "Leaf Logit bit-width"
    }

    # map each x-value -> (n_word, n_int) for tick labeling, using the row
    # with the best acc_quant at that x (ties broken arbitrarily but consistently)
    tick_source = (
        sub.sort_values("acc_quant", ascending=False)
           .drop_duplicates(subset=axis)
    )
    tick_map = {
        row[axis]: (int(row[word_col]), int(row[int_col]))
        for _, row in tick_source.iterrows()
    }

    plt.figure(figsize=(10, 6))
    for depth_name, group in sub.groupby("depth"):
        best = group.groupby(axis)["acc_quant"].max().reset_index()
        best = best.sort_values(axis)
        plt.plot(best[axis], best["acc_quant"], marker="o", label=depth_name)

    baselines = sub.groupby("depth")["acc_float"].first()
    for depth_name, acc in baselines.items():
        plt.axhline(acc, linestyle="--", alpha=0.3)

    x_values = sorted(sub[axis].unique())
    tick_labels = [f"<{tick_map[x][0]},{tick_map[x][1]}>" for x in x_values]
    plt.xticks(x_values, tick_labels) #, rotation=45, ha="right")

    plt.xlabel(f"{axis_labels[axis]}")
    plt.ylabel("Test accuracy")
    plt.title(f"Quantization sweep vs {axis_labels[axis]}")
    plt.legend()
    plt.grid(alpha=0.3)
    #plt.savefig(f"quant_sweep_plot_{label}.png", dpi=150)
    plt.show()

In [ ]:
integer_sweep = pd.read_csv('/eos/home-i00/d/dhnaik/SDT/quant_sweep_results/quant_sweep_results_weight_int_sweep_frac8.csv')
fractional_sweep = pd.read_csv('/eos/home-i00/d/dhnaik/SDT/quant_sweep_results/quant_sweep_results_weight_frac_sweep_int4.csv')
accumulator_frac_sweep = pd.read_csv('/eos/home-i00/d/dhnaik/SDT/quant_sweep_results/quant_sweep_results_accumulator_frac_sweep_int5.csv')
leaf_frac_sweep = pd.read_csv('/eos/home-i00/d/dhnaik/SDT/quant_sweep_results/quant_sweep_results_leaf_frac_sweep_int3.csv')
leaf_int_sweep = pd.read_csv('/eos/home-i00/d/dhnaik/SDT/quant_sweep_results/quant_sweep_results_leaf_int_sweep_frac5.csv')

integer_sweep["acc_n_word"] = 9
integer_sweep["acc_n_frac"] = 4
fractional_sweep["acc_n_word"] = 9
fractional_sweep["acc_n_frac"] = 4
integer_sweep["leaf_n_word"] = 12
integer_sweep["leaf_n_frac"] = 9
fractional_sweep["leaf_n_word"] = 12
fractional_sweep["leaf_n_frac"] = 9
accumulator_frac_sweep["leaf_n_word"] = 12
accumulator_frac_sweep["leaf_n_frac"] = 9

In [ ]:
for ax in ["w_n_word"]:
    plot_sweep(integer_sweep, axis=ax, requant_every_layer=True, label='weight_int_sweep')
    plot_sweep(fractional_sweep, axis=ax, requant_every_layer=True, label='weight_frac_sweep')
for ax in ["acc_n_word"]:
    plot_sweep(accumulator_frac_sweep, axis=ax, requant_every_layer=True, label='accum_frac_sweep')
for ax in ["leaf_n_word"]:
    plot_sweep(leaf_frac_sweep, axis=ax, requant_every_layer=True, label='leaf_frac_sweep')
    plot_sweep(leaf_int_sweep, axis=ax, requant_every_layer=True, label='leaf_int_sweep')